# Week 6 fixed-batch mechanics diagnostics

Run this only after the first controlled suite selects `debug_model_loss_optimizer`. It tests shared lower-learning-rate and loss protocols on the same fixed 16 examples. It does not run rollouts, architecture variants, production training, or OOD evaluation.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from IPython.display import FileLink, display

IMPLEMENTATION_COMMIT = '68a511867ed4f02bbbefe7743ff42799d0fd8a66'
REPOSITORY_URL = 'https://github.com/madhavkapoor13/ChronoPDE.git'
REPOSITORY = Path('/kaggle/working/Chrono_pde')
OUTPUT = REPOSITORY / 'artifacts/diagnostics/week6/mechanics'
EXPECTED_DATA_SIZE = 2_389_261_328
EXPECTED_DATA_SHA256 = (
    '907aa0d79e604e68ce2d4f5cccfd93ffc64eb68c473caf3edae4be438472caec'
)

print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle settings'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
data_candidates = [
    path
    for path in Path('/kaggle/input').rglob('chronopde.h5')
    if path.is_file() and path.stat().st_size == EXPECTED_DATA_SIZE
]
assert data_candidates, 'Attach the private dataset containing chronopde.h5'
DATA = data_candidates[0]
digest = hashlib.sha256()
with DATA.open('rb') as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
assert digest.hexdigest() == EXPECTED_DATA_SHA256
print('Verified dataset:', DATA)


In [ ]:
if REPOSITORY.exists():
    shutil.rmtree(REPOSITORY)
subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
subprocess.run(
    ['git', '-C', str(REPOSITORY), 'checkout', '--detach', IMPLEMENTATION_COMMIT],
    check=True,
)
os.chdir(REPOSITORY)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--ignore-requires-python', '-e', '.'],
    check=True,
)
actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_commit == IMPLEMENTATION_COMMIT
print('Pinned commit:', actual_commit)


In [ ]:
command = [
    sys.executable,
    'scripts/diagnose_mechanics.py',
    '--config',
    'configs/project.yaml',
    '--data-path',
    str(DATA),
    '--device',
    'cuda',
]
print('Launching:', ' '.join(command))
diagnostic = subprocess.run(command, check=False)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'notebook_return_code.json').write_text(
    json.dumps({'return_code': diagnostic.returncode}, indent=2) + '\n'
)
print('Diagnostic return code:', diagnostic.returncode)


In [ ]:
summary_path = OUTPUT / 'suite_summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
    print('Routing decision:', summary['route'])
else:
    print('No suite summary was produced; package partial evidence for debugging.')


In [ ]:
package_root = Path('/kaggle/working/chronopde_week6_mechanics_package')
if package_root.exists():
    shutil.rmtree(package_root)
shutil.copytree(OUTPUT, package_root, ignore=shutil.ignore_patterns('*.pt'))
checkpoint_manifest = []
for checkpoint in sorted(OUTPUT.rglob('*.pt')):
    checkpoint_hash = hashlib.sha256()
    with checkpoint.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            checkpoint_hash.update(chunk)
    checkpoint_manifest.append(
        {
            'path': str(checkpoint.relative_to(OUTPUT)),
            'sha256': checkpoint_hash.hexdigest(),
            'size_bytes': checkpoint.stat().st_size,
        }
    )
(package_root / 'checkpoint_manifest.json').write_text(
    json.dumps(checkpoint_manifest, indent=2) + '\n'
)
package = shutil.make_archive(
    '/kaggle/working/chronopde_week6_mechanics_diagnostics',
    'zip',
    root_dir=package_root,
)
print('Downloadable mechanics package:', package)
display(FileLink(package))
